# 1. OmnipathR metabolomics resources & ID translation

Over the last year OmnipathR gained native accessors for almost every
small-molecule resource we routinely reach for in a metabolomics
workflow: **HMDB**, **RaMP-DB**, **MetalinksDB**, **RECON3D**, the
**Chalmers Sysbio GEMs** (incl. Human-GEM), **STITCH**, and
Reactome's **ChEBI**-keyed pathway export. Plus a generic
`translate_ids()` that uses any of these as a backend.

This script is a buffet — every section below is independent. Pick
the resources that match your downstream question; you don't need to
run them all in order. The final two sections (translation +
ambiguity) work on the ccRCC tissue-metabolomics dataset and apply
regardless of which resource sections you skipped.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(OmnipathR)

## What translation backends does OmnipathR know about?

`id_translation_resources()` lists every resource OmnipathR can use
as a backend for `translate_ids()`. Some return ID-mapping tables
directly (RaMP, HMDB, Chalmers GEM); others (UniProt's UploadLists,
Ensembl) cover proteins / genes.

In [ ]:
id_translation_resources()

## RaMP-DB

RaMP unifies pathway memberships, chemical class hierarchies, and
cross-resource ID maps from KEGG, Reactome, WikiPathways, HMDB,
ChEBI, and LipidMaps. One SQLite download covers everything.

In [ ]:
ramp_tables()

In [ ]:
# Pathway memberships in long form — one row per (analyte, pathway).
ramp_table("analytehaspathway") |> head()

In [ ]:
# The `source` table is the RaMP ID-mapping backbone — it pairs every
# external ID with its internal `rampId`, which is what `translate_ids()`
# joins on under the hood.
ramp_table("source") |> head()

## MetalinksDB

Metabolite ↔ protein interactions curated by the Saez group:
enzymes, transporters, ligand-receptor pairs, and "other" protein-
metabolite contacts.

In [ ]:
metalinksdb_tables()

In [ ]:
# The `edges` table holds every metabolite-protein edge with a `type`
# tag and a confidence score. Filter by type to pull a specific layer.
ml_edges <- metalinksdb_table("edges")
dim(ml_edges)
head(ml_edges)

In [ ]:
# Ligand-receptor edges only.
ml_lr <- subset(ml_edges, type == "lr")
dim(ml_lr)

## RECON3D

A genome-scale metabolic model — useful when you want reactions,
compartments, or enzyme-metabolite relationships rather than pathway
membership. The `extra_hmdb` flag pulls HMDB cross-IDs from the
Virtual Metabolic Human export and joins them onto the metabolite
table.

In [ ]:
recon3d_metabolites() |> head()

In [ ]:
recon3d_reactions() |> head()

In [ ]:
recon3d_compartments()

## Chalmers Sysbio GEMs (Human-GEM and friends)

A family of genome-scale metabolic models maintained by the Chalmers
Sysbio group. Same shape as RECON3D (metabolites + reactions +
genes) but with a different curation pedigree and richer ID coverage.

In [ ]:
chalmers_gem_metabolites() |> head()

In [ ]:
chalmers_gem_reactions() |> head()

In [ ]:
# Cross-IDs between Chalmers' internal "metabolicatlas" labels and
# common public IDs. Powers the `chalmers = TRUE` backend of
# `translate_ids()`.
chalmers_gem_id_mapping_table("metabolicatlas", "hmdb") |> head()

## Reactome ↔ ChEBI

A flat ChEBI-keyed pathway export — handy when you already have
ChEBI IDs and want pathway memberships without going through RaMP.

In [ ]:
reactome_chebi() |> head()

In [ ]:
reactome_chebi_pathways() |> head()

## STITCH

Chemical ↔ protein interactions from STRING-DB's chemistry sister
project. Two complementary tables: `links` (just edges + scores) and
`actions` (mode-of-action annotations).

**Note**: the full STITCH download is large — pass `score_threshold`
/ organism filters to keep it manageable. Skip in the live session
if your VM is short on disk.

In [ ]:
# stitch_links(threshold = 700) |> head()      # uncomment to run
# stitch_actions() |> head()

## HMDB

Human Metabolome Database — the canonical curated metabolite
resource. **Note**: the HMDB XML dump is multi-GB and HMDB.ca
returns HTTP 403 from cloud-VM IP ranges. If you're on the EBI
training-room VM, prefer the RaMP path above (which already imports
the HMDB content). The accessor is shown for completeness.

In [ ]:
hmdb_metabolite_fields() |> head(20)

In [ ]:
# `hmdb_table('metabolites')` triggers the multi-GB download — only
# run this on a workstation with a reliable network and at least
# ~10 GB free. Uncomment to actually fetch.
# hmdb_table("metabolites", fields = c("accession", "name", "kegg_id"))

## Metabolite ID translation with RaMP

`translate_ids()` is OmnipathR's general translator. For metabolites
the recommended backend is **RaMP** — pass `ramp = TRUE` to force it,
or let `entity_type = 'small_molecule'` pick it automatically.

We work on the same ccRCC tissue-metabolomics dataset Christina uses
in script 03 (Hakimi et al. 2018, 138 matched ccRCC / normal pairs)
but stick to vanilla OmnipathR functions for the translation and the
ambiguity counting.

In [ ]:
data(tissue_meta, package = "MetaProViz")
head(tissue_meta)

In [ ]:
# Wrangle: keep only entries with a real metabolite name (drop the
# "X-####" placeholders), normalise the ID separators, and explode
# multi-ID cells into one row per (metabolite, HMDB) pair.
metabolites <- tissue_meta |>
    dplyr::filter(!stringr::str_detect(Metabolite, "^X\\s*-\\s*\\d+$")) |>
    dplyr::mutate(
        HMDB = normalize_id_cell(HMDB),
        KEGG = normalize_id_cell(KEGG),
        PUBCHEM = normalize_id_cell(PUBCHEM),
    ) |>
    dplyr::filter(!is.na(HMDB)) |>
    tidyr::separate_rows(HMDB, sep = ", ") |>
    dplyr::distinct(Metabolite, HMDB)

dim(metabolites)
head(metabolites)

## HMDB → KEGG via RaMP

`translate_ids(d, hmdb_id = hmdb, kegg, ramp = TRUE)` reads
"the column `hmdb_id` holds HMDB IDs; add a new `kegg` column
resolved via RaMP." `keep_untranslated = TRUE` (the default) keeps
rows that didn't resolve so we can quantify the loss.

In [ ]:
hmdb2kegg <- translate_ids(
    metabolites,
    HMDB = hmdb,
    kegg,
    ramp = TRUE,
)
head(hmdb2kegg)

In [ ]:
# How many of our metabolites got at least one KEGG ID?
hmdb2kegg |>
    dplyr::group_by(Metabolite) |>
    dplyr::summarise(any_kegg = any(!is.na(kegg))) |>
    dplyr::count(any_kegg)

## HMDB → ChEBI via RaMP

Same pattern, different target ID type. ChEBI is the lingua franca
of pathway databases (Reactome, WikiPathways) so this translation
unlocks downstream pathway enrichment.

In [ ]:
hmdb2chebi <- translate_ids(
    metabolites,
    HMDB = hmdb,
    chebi,
    ramp = TRUE,
)
head(hmdb2chebi)

## Ambiguity analysis

`ambiguity()` quantifies how many-to-many a translation is — the
core problem in metabolite ID work because chemically near-identical
species (stereoisomers, ionisation states, salt forms) often share
IDs.

- `qualify = TRUE` adds two boolean columns:
  `ambig_<from_col>` (does this `from` value point at multiple `to`s?)
  and `ambig_<to_col>` (does this `to` value receive multiple `from`s?)
- `quantify = TRUE` adds the integer counts.
- `summary = TRUE` returns a tidy one-row summary instead.

In [ ]:
# Per-row ambiguity flags + counts.
amb <- ambiguity(
    hmdb2kegg,
    from_col = HMDB,
    to_col = kegg,
    quantify = TRUE,
    qualify = TRUE,
)
head(amb)

In [ ]:
# One-shot summary: how many HMDB IDs map to multiple KEGGs, and vice
# versa, across the whole table?
ambiguity(
    hmdb2kegg,
    from_col = HMDB,
    to_col = kegg,
    summary = TRUE,
)

In [ ]:
# `translate_ids()` itself can do the same accounting in one pass —
# add `quantify_ambiguity = TRUE` / `ambiguity_summary = TRUE` and
# you get the translation **and** the diagnostics from a single call.
translate_ids(
    metabolites,
    HMDB = hmdb,
    kegg,
    ramp = TRUE,
    ambiguity_summary = TRUE,
)

**Recap.** OmnipathR exposes the major small-molecule resources
behind a uniform accessor pattern (`<resource>_table()` /
`<resource>_<entity>()`), and `translate_ids(..., ramp = TRUE)` plus
`ambiguity()` cover the ID-translation + diagnostics use case end to
end. In script 02 we'll see how MetaProViz wraps these into a
tidier metabolomics-workflow API; in scripts 03–05 we use both
layers in concert.

# 2. MetaProViz prior-knowledge sets

MetaProViz wraps OmnipathR with a tidier, table-in / table-out API
tuned for metabolomics workflows. On top of the raw OmnipathR
accessors (which we tour in script 01) it ships **ready-to-enrich
metabolite-sets** under one umbrella:

- **MetSigDB** — curated metabolite-sets for enrichment analysis
  (KEGG, Reactome, WikiPathways pathway sets, plus chemical-class
  sets and a cancer-specific MaCDB set)
- **Gene-metabolite sets** — Hallmarks and Gaude pathway gene sets
  converted to gene + metabolite memberships
- **MetalinksDB** — wrapped as enzyme / transporter / receptor /
  other-protein-metabolite enrichment tables

Each is fetched on demand and cached locally. `compare_pk()` lets us
look at overlap between any combination of these sets.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)

## 1. MetaProViz MetSigDB
Here we will showcase some content of the Metabolite Signature database (MetSigDB),
which is a curated collection of metabolite sets for enrichment analysis.
It is built on top of OmnipathR and provides ready-to-use tables for pathway-style analyses.

### 1.1. Pathway-metabolite sets
MetaProViz includes KEGG, wikipathwyas, reactome

In [ ]:

# KEGG
KEGG_Pathways <- metsigdb_kegg(exclude_metabolites = "all")# default= all

#Reactome
metsigdb_reactome()

### 1.2. Pathway-gene-metabolite sets
MetaProViz convert gene-sets to gene-metabolite sets and includes two genesets,
Hallmarks and Gaude

In [ ]:

# Gaude
data("gaude_pathways")

gaude_pathways <- make_gene_metab_set( #MetaproViz function
    input_pk =   gaude_pathways,
    metadata_info = c(Target = "gene"),
    pk_name = "gaude_pathways",
    save_table = NULL,
    exclude_metabolites = "all"
)$GeneMetabSet %>%
    mutate(
        gene_metab_source = "gaude_pathways",
        interaction_type = if_else(grepl("^HMDB", feature), "pathway-metabolite", "pathway-metabolic_enzyme")
    )


# Hallmark
data("hallmarks")

hallmarks_pathways <- make_gene_metab_set(
    input_pk = hallmarks,
    metadata_info = c(Target = "gene"),
    pk_name = "hallmarks",
    save_table = NULL,
    exclude_metabolites = "all"
)$GeneMetabSet %>%
    mutate(
        gene_metab_source = "hallmarks",
        interaction_type = if_else(grepl("^HMDB", feature), "pathway-metabolite", "pathway-metabolic_enzyme")
    )

### 1.3. Chemical Class -metabolite sets
Based on RaMP-2.0 calssification using CallyFire

In [ ]:

# Chemical class
ChemicalClass <- metsigdb_chemicalclass()

### 1.4. Cancer-metabolite sets
Based on macDB content

In [ ]:

# Cancer
Cancer <- metsigdb_macdb()



### 1.5. Metabolite-receptor, -transporter, -enzyme sets from MetalinksDB
MetaProViz wraps MetalinksDB into a ready-to-enrich pathway-style table
via `metsigdb_metalinks()`:

In [ ]:
ml_pk <- metsigdb_metalinks()
head(ml_pk)

ml_pk_expansion <- tibble::as_tibble(ml_pk)
unknown <- unique(ml_pk_expansion[["interaction_family"]][!ml_pk_expansion[["interaction_family"]] %in% c(
    "Other protein-metabolite",
    "Transporter-metabolite",
    "Receptor-metabolite",
    "Enzyme-metabolite"
)])
unknown <- unknown[!is.na(unknown)]

ml_pk_expansion <- list(
    metsigdb_metalinks = ml_pk_expansion,
    metsigdb_metalinks__other_protein_metabolite =
        dplyr::filter(ml_pk_expansion, interaction_family == "Other protein-metabolite"),
    metsigdb_metalinks__transporter_metabolite =
        dplyr::filter(ml_pk_expansion, interaction_family == "Transporter-metabolite"),
    metsigdb_metalinks__receptor_metabolite =
        dplyr::filter(ml_pk_expansion, interaction_family == "Receptor-metabolite"),
    metsigdb_metalinks__enzyme_metabolite =
        dplyr::filter(ml_pk_expansion, interaction_family == "Enzyme-metabolite")
)

## Compare PK coverage

Are the same metabolites covered by all of these? Use
`compare_pk()` to see the intersection / specificity.

In [ ]:
pk_comp_res <- compare_pk(data = list(Hallmarks = as.data.frame(hallmarks_pathways),
                                      Gaude = as.data.frame(gaude_pathways),
                                      MetalinksDB = as.data.frame(ml_pk))
)

**Recap.** MetaProViz wraps OmnipathR with a tidy, table-in / table-out
API tuned for metabolomics workflows: it ships ready-to-enrich
pathway-, gene-, chemical-class-, and metalinks-derived metabolite-sets
under one umbrella, plus `compare_pk()` to inspect overlap. The
underlying OmnipathR resources (MetalinksDB, RaMP, RECON3D, …) are
demonstrated directly in script 01.

# 3. MetaProViz Metabolite ID workflow
This is important to improving the connection between prior knowledge and
metabolomics features.
Many databases that collect metabolite information, such as the Human
Metabolome Database (HMDB), include multiple entries for the same metabolite
with different degrees of ambiguity. This poses a difficulty when assigning
metabolite IDs to measured data where e.g. stereoisomers are not distinguished.
Hence, if detection is unspecific, it is crucial to assign all possible
IDs to increase the overlap with the prior knowledge.
MetaProViz offers methodologies to solve such conflicts to allow robust mapping
of experimental data with prior knowledge. In detail, MetaProViz performs
feature space quality control, offers functionalities to increase the
metabolite ID feature space, including ID translation, ID traversion through
a metabolite ID graph and enantiomer addition and quantifies mapping ambiguities.
this is the workflow we will follow in the next steps.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))

library(MetaProViz)

## 0. Load the data

We work with publicly available metabolomic profiling on 138 matched clear cell
renal cell carcinoma (ccRCC)/normal tissue pairsdownloaded from
https://www.cell.com/cancer-cell/comments/S1535-6108(15)00468-7. Here we are
interested in processing and working with the metababolite feature space and
will only load the feature metadata for exploration.

FYI: Since v3.99.10 (Oct 2025) MetaProViz also uses `SummarizedExperiment` (SE) as
its canonical container — the same object Bioconductor's omics packages
expect. Each row is a metabolite; each column is a sample. Sample
annotations live in `colData(se)`, feature annotations in `rowData(se)`,
the actual measurements in `assays(se)`. You can also load all data in se format
using `data(tissue_norm_se)`

Check which type of metabolite IDs are available and what other information
where provided by the authors.

In [ ]:

data(tissue_meta)

head(tissue_meta)

## Data wrangling
It is important to inspect the separator of the metabolite ids and unify them,
as it is common that are not unified. In this case, we have some entries with ";"
and some with "," as separator. We will unify them to "," and save the cleaned
object for later use using a helper from utils.

In [ ]:

MetaboliteIDs <- tissue_meta %>%
    dplyr::filter(!stringr::str_detect(Metabolite, "^X\\s*-\\s*\\d+$"))%>%# only keep entries with trivial names
    dplyr::mutate(
        SUPER_PATHWAY = dplyr::coalesce(SUPER_PATHWAY, "None"),
        SUB_PATHWAY   = dplyr::coalesce(SUB_PATHWAY, "None"),
        CAS = normalize_id_cell(CAS),
        HMDB = normalize_id_cell(HMDB),
        KEGG = normalize_id_cell(KEGG),
        PUBCHEM = normalize_id_cell(PUBCHEM)
    ) %>%
    mutate(across(where(is.character), ~ gsub(";", ",", .x)))

# save .rds file
saveRDS(MetaboliteIDs, file.path(mp_results_dir(), "MetaboliteIDs_clean_idseparator.rds"))

**Recap.** We loaded the feature metadata of the ccRCC dataset published by
Hakimi et al. in 2018 and inspected the metabolite IDs available and ensured
the metabolite ID separator is unified and saved the cleaned object.

## 1. Feature space quality control

Analyse the existent metabolite ID space using MetaProViz `compare_pk()`
function to check the overlap of the different ID types and the coverage of
the feature space and `count_id()`.

In [ ]:

# compare id space
ccRCC_CompareIDs <- compare_pk(data = list(Biocft = MetaboliteIDs |>
                                               dplyr::rename("Class"="SUPER_PATHWAY")),
                               name_col = "Metabolite",
                               metadata_info = list(Biocft = c("KEGG", "HMDB", "PUBCHEM")),
                               plot_name = "Overlap of ID types in ccRCC data")

Here we notice that 76 features have no metabolite ID assigned, yet have a
trivial name and a metabolite class assigned. For 135 metabolites we only
have a pubchem ID, yet no HMDB or KEGG ID. Only 43% of all features have HMDB,
KEGG and Pubchem IDs assigned, whilst 23% only had a PubChem ID assigned.
One explanation could be that the databases covered less structures when the
study was published in 2016.

In [ ]:

#count ids
# 1. HMDB:
Plot1_HMDB <- count_id(MetaboliteIDs,
                       delimiter = ", ",
                       "HMDB")

# The output is a data table that is visualized in a barplot
head(Plot1_HMDB[["Table"]])
Plot1_HMDB[["Plot_Sized"]]

# 2. KEGG:
Plot1_KEGG <- count_id(MetaboliteIDs,
                       delimiter = ", ",
                       "KEGG")

# 3. PubChem:
Plot1_PubChem <- count_id(MetaboliteIDs,
                          delimiter = ", ",
                          "PUBCHEM")

Now we can extract some summary statistics from the count_id() output to get a
better overview of the ID space and how it changes with the different steps of
the workflow. For this we use our helper functions established ain our utils.

This table stores one row per database and category, for example:
HMDB_Original, KEGG_cleaned, PUBCHEM_Original, etc.

Column meanings:
no_ID         = number of features with no ID in that database
Single_ID     = number of features with exactly one ID in that database
Multiple_IDs  = number of features with more than one ID in that database
Total_IDs     = total number of IDs assigned in that database
                calculated as sum(entry_count)

In [ ]:

id_count_df <- tibble(
    row_name      = character(),
    no_ID         = integer(),
    Single_ID     = integer(),
    Multiple_IDs  = integer(),
    Total_IDs     = integer()
)

id_count_df <- append_id_counts(id_count_df, Plot1_HMDB,    "HMDB_Original")
id_count_df <- append_id_counts(id_count_df, Plot1_KEGG,    "KEGG_Original")
id_count_df <- append_id_counts(id_count_df, Plot1_PubChem, "PUBCHEM_Original")

head(id_count_df)

Part of the ID QC is also to check the MetaboliteOD compatibility. This checks
for ID mismatches of features with metabolite IDs that pointed to different
metabolite structures.

In [ ]:

# Note: The input df does not contain ChEBI IDs. For the compatibility check,
# ChEBI IDs are internally used as possible stepstones for ID compatibility
MetaboliteIDs_compatibility_check <- seed_id_compatibility_check(
    data = MetaboliteIDs,
    id_types = c("HMDB", "KEGG", "CHEBI", "PUBCHEM"),
    delimiter = ","
)

MetaboliteIDs_pair_compatibility <- MetaboliteIDs_compatibility_check$ID_pair_compatibility
MetaboliteIDs_data_with_compatibility <- MetaboliteIDs_compatibility_check$data_with_compatibility

### i) Let's retrieve the fully compatible features
For those features no issues were found and hence everything is correct.

In [ ]:

fully_compatible_metabolites <- MetaboliteIDs_data_with_compatibility[MetaboliteIDs_data_with_compatibility$all_seed_ids_compatible == TRUE, "Metabolite"]

# retrieve the id_pairs for these fully_compatibly features
fully_compatible_id_pair_rows <-
    MetaboliteIDs_pair_compatibility %>%
    filter(
        Metabolite %in% unlist(fully_compatible_metabolites)
    )%>%
    mutate(
        manual_correction = FALSE,
        correction_tbl = NA
    )

cat("Distinct features in fully_compatible_id_pair_rows:",
    length(unique(fully_compatible_id_pair_rows$Metabolite)),
    "(checks out)\n")

### ii) Get lost metabolites
keep all direct and secondary matches and discard no_match

In [ ]:
MetaboliteIDs_pair_compatibility_filtered <- MetaboliteIDs_pair_compatibility %>%
    filter(
        (all_seed_ids_compatible == TRUE) |
            (all_seed_ids_compatible == FALSE & compatibility_path == "direct") |
            (all_seed_ids_compatible == FALSE & compatibility_path == "secondary")
    )

lost_Metabolites_after_filtering <- list(
    setdiff(
        unique(MetaboliteIDs_pair_compatibility$Metabolite),
        unique(MetaboliteIDs_pair_compatibility_filtered$Metabolite)
    )
)

lost_id_pair_rows <- MetaboliteIDs_pair_compatibility[ MetaboliteIDs_pair_compatibility$Metabolite
                                                       %in% lost_Metabolites_after_filtering[[1]]  , ]

cat("Distinct metabolites before compatibility filtering:",
    length(unique(MetaboliteIDs_pair_compatibility$Metabolite)),
    "\n")
cat("Distinct metabolites after compatibility filtering:",
    length(unique(MetaboliteIDs_pair_compatibility_filtered$Metabolite)),
    "\n")

# Lets explore some cases:
# Example 1: N-acetyltryptophan
Trypt <- MetaboliteIDs_data_with_compatibility%>%
    filter(Metabolite== "N-acetyltryptophan")
Trypt # Kegg C03137 is D- and Pubchem 700653 is L form.

# as we do not have the time for a manual review in this session you will need to
# trust our review and just filter out the below metabolites:

lost_id_pair_rows_corrected <- lost_id_pair_rows %>%
    mutate(
        KEGG = case_when(
            # drop_id is a helper in utils to Remove one ID from a delimiter-separated cell;
            # return NA if empty.
            Metabolite == "androsterone sulfate" ~ drop_id(KEGG, "C00523"),
            Metabolite == "N-acetylthreonine" ~ drop_id(KEGG, "C01118"),
            Metabolite == "catechol sulfate" ~ drop_id(KEGG, "C00090"),
            TRUE ~ KEGG
        ),
        HMDB = case_when(
            Metabolite == "fructose-6-phosphate" ~ drop_id(HMDB, "HMDB0000124"),
            Metabolite == "galactose" ~ drop_id(HMDB, "HMDB0000143"),
            TRUE ~ HMDB
        ),
        PUBCHEM = case_when(
            Metabolite == "mannose" ~ drop_id(PUBCHEM, "161658"),
            Metabolite == "tyrosylalanine" ~ "7020632",
            TRUE ~ PUBCHEM
        ),
        manual_correction = TRUE,
        correction_tbl = "manual_fix_for_lost_id_pair_rows"
    )

### iii) Get the permutation_df for the partially_compatible features
keep all direct and secondary matches and discard no_match
We will remove 41 partially incompatible cases (= some IDs are connected in
the network whilst others are not

In [ ]:

partially_compatible_metabolites <-
    MetaboliteIDs_data_with_compatibility %>%
    filter(
        all_seed_ids_compatible == FALSE
    ) %>%
    filter(
        ! Metabolite %in% unlist(lost_Metabolites_after_filtering)
    )

cat("Partially compatible metabolite names:",
    length(unique(partially_compatible_metabolites$Metabolite)),
    "\n")

# get the id_pair compatibility for those
partially_compatible_id_pair_rows <-
    MetaboliteIDs_pair_compatibility %>%
    filter(
        Metabolite %in% partially_compatible_metabolites$Metabolite
    )

partially_compatible_id_pair_rows %<>%
    filter(
        compatibility_path != "no_match"   #only keep matches
    ) %>%
    mutate(
        manual_correction = TRUE,
        correction_tbl = "keeping_only_compatible_for_partially_compatible_features"
    )

In [ ]:

# now lets summarize them again, we completely ignore the incompatible features
combined_tables <-
    rbind(
        partially_compatible_id_pair_rows,
        fully_compatible_id_pair_rows,
        lost_id_pair_rows_corrected
    )

# join back together based on "original_row_id", summarizing the IDs per type
MetaboliteIDs_cleaned <- combined_tables %>%
    group_by(original_row_id) %>%
    summarise(
        Metabolite    = first(Metabolite),
        CAS           = first(CAS),
        SUPER_PATHWAY = first(SUPER_PATHWAY),
        SUB_PATHWAY   = first(SUB_PATHWAY),
        COMP_ID       = first(COMP_ID),
        PLATFORM      = first(PLATFORM),
        RI            = first(RI),
        MASS          = first(MASS),

        PUBCHEM = {
            x <- trimws(as.character(PUBCHEM))
            x[x == ""] <- NA_character_
            x <- unique(na.omit(x))
            if (length(x) == 0) NA_character_ else paste(x, collapse = ", ")
        },

        KEGG = {
            x <- trimws(as.character(KEGG))
            x[x == ""] <- NA_character_
            x <- unique(na.omit(x))
            if (length(x) == 0) NA_character_ else paste(x, collapse = ", ")
        },

        HMDB = {
            x <- trimws(as.character(HMDB))
            x[x == ""] <- NA_character_
            x <- unique(na.omit(x))
            if (length(x) == 0) NA_character_ else paste(x, collapse = ", ")
        },

        CHEBI = {
            x <- trimws(as.character(CHEBI))
            x[x == ""] <- NA_character_
            x <- unique(na.omit(x))
            if (length(x) == 0) NA_character_ else paste(x, collapse = ", ")
        },

        manual_correction = first(manual_correction),
        correction_tbl = first(correction_tbl),

        .groups = "drop"
    ) %>%
    select(
        Metabolite,
        CAS,
        SUPER_PATHWAY,
        SUB_PATHWAY,
        COMP_ID,
        PLATFORM,
        RI,
        MASS,
        PUBCHEM,
        KEGG,
        HMDB,
        CHEBI,
        manual_correction,
        correction_tbl
    )

head(MetaboliteIDs_cleaned)

# lets check number of IDs and add to our summary table
ccRCC_CompareIDs_cleaned <- MetaProViz::compare_pk(data = list(Biocft = MetaboliteIDs_cleaned |> dplyr::rename("Class"="SUPER_PATHWAY")),
                                                   name_col = "Metabolite",
                                                   metadata_info = list(Biocft = c("KEGG", "HMDB", "PUBCHEM")),
                                                   plot_name = "Overlap of ID types in ccRCC data")


Plot2_HMDB <- count_id(MetaboliteIDs_cleaned,
                       delimiter = ", ",
                       "HMDB")

# 2. KEGG:
Plot2_KEGG <- count_id(MetaboliteIDs_cleaned,
                       delimiter = ", ",
                       "KEGG")

# 3. PubChem:
Plot2_PubChem <- count_id(MetaboliteIDs_cleaned,
                          delimiter = ", ",
                          "PUBCHEM")


# Add to our summary table:
# cleaned
id_count_df <- append_id_counts(id_count_df, Plot2_HMDB,    "HMDB_Cleaned")
id_count_df <- append_id_counts(id_count_df, Plot2_KEGG,    "KEGG_Cleaned")
id_count_df <- append_id_counts(id_count_df, Plot2_PubChem, "PUBCHEM_Cleaned")

## Calculate delta rows
delta_df <- calculate_id_deltas(id_count_df, "Original", "Cleaned")
delta_df

## 2. Translate Cas to metabolite ID
Features with no HMDB, KEGG or PubChem ID could be translated from the CAS numbers
as there are some cases in which we have a CAS number, yet no HMDB, PubChem
or KEGG ID. We can use the CAS number to check if we can find a metabolite ID
for these features.

In [ ]:

# Extract features with No HMDB, KEGG or PubChem ID:
features_no_ids <- MetaboliteIDs_cleaned %>%
    dplyr::filter(
        (is.na(HMDB)    | trimws(HMDB) %in% c("", "NA")) &
            (is.na(KEGG)    | trimws(KEGG) %in% c("", "NA")) &
            (is.na(PUBCHEM) | trimws(PUBCHEM) %in% c("", "NA"))
    )

#Of those some may have a CAS number:
features_no_ids_with_cas <- features_no_ids %>%
    dplyr::filter(!(is.na(CAS) | trimws(CAS) %in% c("", "NA")))


print(paste0("We have ", nrow(features_no_ids_with_cas),
             " features with CAS number but no other metabolite ID and ",
             nrow(features_no_ids)- nrow(features_no_ids_with_cas),
             " features with no CAS number and no other metabolite ID."))

# translate IDs using MetaProViz
features_ids_from_cas <- translate_id(data = features_no_ids_with_cas,
                                      metadata_info = list(InputID = "CAS",
                                                           grouping_variable = "SUPER_PATHWAY"),
                                      from = "cas",
                                      to = c("hmdb", "kegg", "pubchem"))

# Keep only rows where at least one new ID was actually found
cas_ids_found <- features_ids_from_cas[["TranslatedDF"]] %>%
    dplyr::filter(
        !(is.na(hmdb)    | hmdb    == "") |
            !(is.na(kegg)    | kegg    == "") |
            !(is.na(pubchem) | pubchem == "")
    ) %>%
    # Replace the (empty) original ID columns with the translated ones
    dplyr::mutate(
        HMDB = normalize_id_cell(hmdb),
        KEGG = normalize_id_cell(kegg),
        PUBCHEM = normalize_id_cell(pubchem)
    ) %>%
    dplyr::select(-hmdb, -kegg, -pubchem)

# Merge back: update matching rows in MetaboliteIDs by COMP_ID
MetaboliteIDs_Translated <- MetaboliteIDs_cleaned %>%
    dplyr::rows_update(cas_ids_found, by = "COMP_ID")


# Now we can repeat the `count_id` and `compare_pk` plot after every change to
# the number of metabolite IDs
ccRCC_CompareIDs_Translated <- MetaProViz::compare_pk(data = list(Biocft = MetaboliteIDs_Translated |> dplyr::rename("Class"="SUPER_PATHWAY")),
                                                      name_col = "Metabolite",
                                                      metadata_info = list(Biocft = c("KEGG", "HMDB", "PUBCHEM")),
                                                      plot_name = "Overlap of ID types in ccRCC data")


Plot3_HMDB <- count_id(MetaboliteIDs_Translated,
                       delimiter = ", ",
                       "HMDB")

# 2. KEGG:
Plot3_KEGG <- count_id(MetaboliteIDs_Translated,
                       delimiter = ", ",
                       "KEGG")

# 3. PubChem:
Plot3_PubChem <- count_id(MetaboliteIDs_Translated,
                          delimiter = ", ",
                          "PUBCHEM")

# add translated to our summarized DF
id_count_df <- append_id_counts(id_count_df, Plot3_HMDB,    "HMDB_Translated")
id_count_df <- append_id_counts(id_count_df, Plot3_KEGG,    "KEGG_Translated")
id_count_df <- append_id_counts(id_count_df, Plot3_PubChem, "PUBCHEM_Translated")

## Calculate delta rows
delta_df <- rbind(
    delta_df,
    calculate_id_deltas(id_count_df, "Original", "Translated"),
    calculate_id_deltas(id_count_df, "Cleaned", "Translated")
)

## 3. Traverse metabolite IDs
 we fill the gaps by expanding the metabolite identifiers across HMDB, KEGG
and PubChem by building and traversing a metabolite ID graph. This fills the
gaps by iteratively going through the cross-database mappings and collecting
all connected IDs whilst additionally adding ChEBI IDs to the features.

In [ ]:

# Harmonize ID separators for traversal (function expects one delimiter)
Input_Traverse <- MetaboliteIDs_Translated %>%
    dplyr::mutate(
        HMDB = normalize_id_cell(HMDB),
        KEGG = normalize_id_cell(KEGG),
        PUBCHEM = normalize_id_cell(PUBCHEM)
    )

Results_Traverse <- MetaProViz::traverse_ids(
    data = Input_Traverse,
    id_types = c("HMDB", "KEGG", "CHEBI", "PUBCHEM"),
    delimiter = ",",
    path = "MetaProViz_Results/PK/Traverse"
)

## inspect results
traverse_ID_pair_compatibility <- Results_Traverse$ID_pair_compatibility
traverse_ID_edges <- Results_Traverse$ID_Edges_prior_knowledge
traverse_ID_ExpandedDF <- Results_Traverse$ExpandedDF

## how many features of the traverse_ID_ExpandedDF are all_seed_ids_compatible?
cat("Fully compatible features (PUBCHEM/KEGG/HMDB):",
    length(which(traverse_ID_ExpandedDF$all_seed_ids_compatible == TRUE)),
    "\n")

## how many features of the traverse_ID_ExpandedDF are NOT all_seed_ids_compatible?
cat("NOT fully compatible features (PUBCHEM/KEGG/HMDB):",
    length(which(traverse_ID_ExpandedDF$all_seed_ids_compatible == FALSE)),
    "\n")

## how many of these features with at least some incompatibility were NOT manually curated?
cat("NOT fully compatible features (PUBCHEM/KEGG/HMDB) WITHOUT a 'manual_curation' tag:",
    length(which(traverse_ID_ExpandedDF$all_seed_ids_compatible == FALSE &
                     traverse_ID_ExpandedDF$manual_curation == FALSE)),
    "\n")

# Incorporate the newly added features into a new df "MetaboliteIDs_traverse"
MetaboliteIDs_traverse <-
    traverse_ID_ExpandedDF %>%
    mutate(
        PUBCHEM = {
            PUBCHEM_translated %>%
                normalize_id_cell() %>%
                gsub("CID", "", .)
        },
        KEGG = KEGG_translated %>% normalize_id_cell(),
        HMDB = HMDB_translated %>% normalize_id_cell(),
        CHEBI = CHEBI_translated %>% normalize_id_cell()
    ) %>%
    select(
        - all_seed_ids_compatible,
        - row_id,
        - HMDB_translated,
        - KEGG_translated,
        - CHEBI_translated,
        - PUBCHEM_translated,
        - n_seed_ids,
        - n_HMDB_translated,
        - n_KEGG_translated,
        - n_CHEBI_translated,
        - n_PUBCHEM_translated,
        - mapping_expanded,
        - ambiguous_seed,
        - large_mapping
    )


# Now we can repeat the `count_id` and `compare_pk` plot after every change to
# the number of metabolite IDs
ccRCC_CompareIDs_Traverse <- MetaProViz::compare_pk(data = list(Biocft = MetaboliteIDs_traverse |> dplyr::rename("Class"="SUPER_PATHWAY")),
                                                    name_col = "Metabolite",
                                                    metadata_info = list(Biocft = c("KEGG", "HMDB", "PUBCHEM")),
                                                    plot_name = "Overlap of ID types in ccRCC data")


Plot4_HMDB <- count_id(MetaboliteIDs_traverse,
                       delimiter = ", ",
                       "HMDB")

# 2. KEGG:
Plot4_KEGG <- count_id(MetaboliteIDs_traverse,
                       delimiter = ", ",
                       "KEGG")

# 3. PubChem:
Plot4_PubChem <- count_id(MetaboliteIDs_traverse,
                          delimiter = ", ",
                          "PUBCHEM")

# 3. Chebi:
Plot4_CHEBI <- count_id(MetaboliteIDs_traverse,
                        delimiter = ", ",
                        "CHEBI")

# extend counting dataframe with traverse_id results
# traverse
id_count_df <- append_id_counts(id_count_df, Plot4_HMDB,    "HMDB_Traverse")
id_count_df <- append_id_counts(id_count_df, Plot4_KEGG,    "KEGG_Traverse")
id_count_df <- append_id_counts(id_count_df, Plot4_PubChem, "PUBCHEM_Traverse")
id_count_df <- append_id_counts(id_count_df, Plot4_CHEBI, "CHEBI_Traverse")

# Calculate delta rows
delta_df <- rbind(
    delta_df,
    calculate_id_deltas(id_count_df, "Original", "Traverse"),
    calculate_id_deltas(id_count_df, "Cleaned", "Traverse"),
    calculate_id_deltas(id_count_df, "Translated", "Traverse")
)

## 4. Equivalent IDs
If you do not know if you expect L- or D- of your aminoacids or R/S sugar,
which both are not distinguishable by LC-MS, you can add equivalent IDs.
As we have human samples and do not want the D-As form, we will skip this step.
You can see details here:
https://saezlab.github.io/MetaProViz/reference/equivalent_id.html

In [ ]:

saveRDS(id_count_df, file.path(mp_results_dir(), "id_count_df.rds"))
saveRDS( delta_df, file.path(mp_results_dir(), "delta_df.rds"))
saveRDS(MetaboliteIDs_traverse, file.path(mp_results_dir(), "MetaboliteIDs_expanded.rds"))

## 5. Explore how our ID space changed:

In [ ]:

#Compare plots:
ccRCC_CompareIDs
ccRCC_CompareIDs_Traverse


#Check result tables:
id_count_df

delta_df

plot_df <- id_count_df %>%
    mutate(
        db = sub("_.*$", "", row_name),
        stage = sub("^[^_]+_", "", row_name),
        db = factor(db, levels = c("HMDB", "KEGG", "PUBCHEM", "CHEBI")),
        stage = factor(stage, levels = c("Original", "Cleaned", "Translated", "Traverse"))
    ) %>%
    arrange(db, stage)

ggplot(plot_df, aes(x = db, y = Total_IDs, fill = stage)) +
    geom_col(position = position_dodge(width = 0.8), width = 0.7) +
    labs(x = NULL, y = "Total IDs", fill = NULL) +
    theme_minimal(base_size = 12)

**Recap.** MetaProViz offers a workflow to compile the ID feature space prior to
enrichment analysis and mapping to PK.

# 4. Integrate feature IDs with prior knowledge and analyse mapping ambiguities
between our feature ID space and the prior-knowledge resource of choice

Most prior-knowledge resources speak different ID dialects (HMDB, ChEBI,
KEGG, PubChem, LipidMaps, …). Before we can connect our differential analysis
results to pathways, we need to check for mapping issues that potentially need
to be solved.


In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))

library(MetaProViz)

## 1. Load data and prior knowledge.

We will look at different prior knowledge resources in the next section, but
for now we will load the cleaned feature metadata object we created and KEGG
pathways as prior knowledge

In [ ]:

# Load our feature metadata:
MetaboliteIDs_expanded <- readRDS(file.path(mp_results_dir(), "MetaboliteIDs_expanded.rds"))

Tissue_MetaData_Extended <- MetaboliteIDs_expanded%>%
    as.data.frame(
        MetaboliteIDs_addEquivalent
    ) %>%
    select(
        Metabolite,
        KEGG,
        manual_correction,
        correction_tbl
    )

# Load KEGG pathways from MetaProViz
KEGG_Pathways <- metsigdb_kegg()

## 2. Analyse mapping ambiguities between our feature ID space and the
prior-knowledge resource of choice

In [ ]:

#check mapping with metadata using MetaProViz
ccRCC_to_KEGGPathways <- checkmatch_pk_to_data(data = Tissue_MetaData_Extended,
                                               input_pk = KEGG_Pathways,
                                               metadata_info = c(InputID = "KEGG",
                                                                 PriorID = "MetaboliteID",
                                                                 grouping_variable = "term"))


# inspect results
ccRCC_to_KEGGPathways_data_summary <- as.data.frame(ccRCC_to_KEGGPathways$data_summary)
ccRCC_to_KEGGPathways_GroupingVariable_summary <- as.data.frame(ccRCC_to_KEGGPathways$GroupingVariable_summary)

# Add to our input
ccRCC_to_KEGGPathways_data_summary <-
    Tissue_MetaData_Extended %>%
    select(
        Metabolite,
        KEGG,
        manual_correction,
        correction_tbl
    ) %>%
    left_join(
        ccRCC_to_KEGGPathways_data_summary,
        by = "KEGG"
    )

# convert lists inside ccRCC_to_KEGGPathways_data_summary to strings
ccRCC_to_KEGGPathways_data_summary[] <- lapply(
    ccRCC_to_KEGGPathways_data_summary,
    function(x) {
        if (is.list(x)) sapply(x, toString) else x
    }
)


#Lets look at the problematic terms:
problems_terms <- ccRCC_to_KEGGPathways_GroupingVariable_summary %>%
    filter(!Group_Conflict_Notes== "None")

print(problems_terms[,c("KEGG", "original_count", "matches_count", "MetaboliteID", "term")])

Dependent on the biological question and the organism and prior knowledge,
one can either maintain the metabolite ID of the more likely metabolite
(e.g. in human its more likely that we have L-aminoacid than D-aminoacid) or
the metabolite ID that is represented in more/less pathways (specificity).
If we are looking into the cases where we do have multiple IDs, for cases with
no match to the prior knowledge we can just maintain one ID, whilst for cases
with exactly one match we should maintain the ID that is found in the prior knowledge.

In case of `ActionRequired=="Check"`, we can look into the column `Action_Specific`
which contains additional information. In case of the entry `KeepEachID`,
multiple matches to the prior knowledge were found, yet the features are in
different pathways (=GroupingVariable). Yet, in case of `KeepOneID`,
the different IDs map to the same pathway in the prior knowledge for at least
one case and therefore keeping both would inflate the enrichment analysis.

In [ ]:

SelectedIDs <- ccRCC_to_KEGGPathways_data_summary %>%
    #Expand rows where Action == KeepEachID by splitting `matches`
    mutate(matches_split = if_else(Action_Specific == "KeepEachID", matches, NA_character_)) %>%
    # separate_rows(matches_split, sep = ",\\s*") %>%
    mutate(
        InputID_select = if_else(
            Action_Specific  == "KeepEachID",
            matches_split,
            InputID_select
        )
    ) %>%
    select(-matches_split) %>%
    mutate(  # remember IDs before to compare at the end of this pipe chain
        InputID_before = InputID_select
    ) %>%
    #Select one ID for AcionSpecific==KeepOneID
    dplyr::mutate(
        InputID_select = case_when(
            Action_Specific == "KeepOneID" & matches ==  "C00025, C00217, C00302" ~ "C00025", # L-Glutamate vs D-Glutamate vs Glutamate compounds. Keep L-Glutamate, most prevalent in KEGG pathways
            Action_Specific == "KeepOneID" & matches ==  "C00065, C00716, C00740" ~ "C00065", # L-Serine vs Serine vs D-Serine
            Action_Specific == "KeepOneID" & matches ==  "C00072, C01041"         ~ "C00072", # Ascorbate vs Monodehydroascorbate. almost same molecule, just one H molecule more in Ascorbate. Keep C00072 since in more pathways
            Action_Specific == "KeepOneID" & matches ==  "C00077, C00515"         ~ "C00077", # L-Ornithine vs D-Ornithine. Keep L-Ornithine
            Action_Specific == "KeepOneID" & matches ==  "C00092, C00668"         ~ "C00092, C00668", # D-Glucose 6-phosphate vs alpha-D-Glucose 6-phosphate. Both present in lots of KEGG_Pathways, with a very low intersection of pathway terms between them. 2 shared pathways and 16 non-shared ones. Remove one of the KEGG IDs from pathways which include both
            Action_Specific == "KeepOneID" & matches ==  "C00221, C00031, C00267" ~ "C00031", # These are D- and L-Glucose and alpha-D-Glucose. We have human samples, so in this conflict we will maintain L-Glucose
            Action_Specific == "KeepOneID" & matches ==  "C00258, C01921"         ~ "C00258", # D-Glycerate vs Glycocholate. This is an ID mismatch, since they are clearly very different molecules. C01921 was added at some translation step, discard it
            Action_Specific == "KeepOneID" & matches ==  "C00309, C00508"         ~ "C00309", # D-Ribulose vs L-Ribulose. Keep C00309, since both map to the same two pathways, so it is irrelevant
            Action_Specific == "KeepOneID" & matches ==  "C00379, C01904"         ~ "C00379", # Xylitol vs D-Arabitol/D-Lyxitol. Stereochemic difference only.  C00379 maps to one more pathway
            Action_Specific == "KeepOneID" & matches ==  "C00474, C00532, C01904" ~ "C00474", # Ribitol/Adonitol vs L-Arabitol vs D-Arabitol. C00474 maps to one more pathway
            Action_Specific == "KeepOneID" & matches ==  "C00502, C05411"         ~ "C05411", # D-Xylonate vs L-Xylonate. Both map to two pathways, one overlapping. We keep L
            Action_Specific == "KeepOneID" & matches ==  "C01087, C02630"         ~ "C01087, C02630", # (R)-2-Hydroxyglutarate vs 2-Hydroxyglutarate. Both map to three pathways, one overlapping. --> Remove (R)-2-Hydroxyglutarate from KEGG pathway "Metabolic Pathways"
            Action_Specific == "KeepOneID" & matches ==  "C03242, C16522"         ~ "C03242", # Dihomo-gamma-linolenate vs Icosatrienoic acid. Both fatty acids with different double bonds. C03242 maps to two more pathways.
            Action_Specific == "KeepOneID" & matches ==  "C03460, C03722"         ~ "C03722", # 2-Methylprop-2-enoyl-CoA versus Quinolinate. No evidence, hence we keep the one present in more pathways (C03722=7 pathways, C03460=2 pathway)
            Action_Specific == "KeepOneID" & matches ==  "C05793, C05794"         ~ "C05793", # L-Urobilin vs Urobilin. C05793 was in originally assigned, keep it.
            Action_Specific == "KeepOneID" & matches ==  "C17737, C00695"         ~ "C00695", # Allocholic acid versus Cholic acid. No evidence, hence we keep the one present in more pathways (C00695 = 4 pathways, C17737 = 1 pathway)
            Action_Specific == "KeepOneID" ~ InputID_select,  # Keep NA where not matched manually
            TRUE ~ InputID_select
        ),
        corrected_during_checkmatch = is.na(InputID_before) ## flag whether changed above
    ) %>%
    select(  ## remove temp column
        -InputID_before
    )

We identified that in some cases in our data, there are two MetaboliteIDs with e.g.
stereochemical differences but they map to multiple pathways. Therefore, we are
going to remember them here and later remove them from the Pathway PK

Remove (R)-2-Hydroxyglutarate (C01087) from KEGG pathway "Metabolic Pathways"
C00092, C00668: remove C00668 from Pathways which contain both of them

Lastly, we need to add the column including our selected IDs to the metadata table.

In [ ]:

Tissue_MetaData_Extended_cleaned <-
    merge(
        x = SelectedIDs %>%
            select(Metabolite, InputID_select, corrected_during_checkmatch),
        y = Tissue_MetaData_Extended,
        by = "Metabolite",
        all.y = TRUE
    ) %>%
    select(
        Metabolite, KEGG, InputID_select, manual_correction, correction_tbl, corrected_during_checkmatch
    )


saveRDS(Tissue_MetaData_Extended_cleaned, file.path(mp_results_dir(), "MetaboliteIDs_expanded_cleanedKEGG.rds"))

**Recap.** We created final table that will be used as input for the
enrichment analysis using KEGG as prior knowledge

# 5. Enrichment analysis

- **`standard_ora()`** — over-representation analysis: are the
  significantly altered metabolites enriched in any metabolite-set Fisher's
  exact test, the same engine `clusterProfiler` uses.


- **`cluster_ora()`** - requires clutsers of metabolites you obtain from other
previous analysis (e.g. biological regulated clustering) and tests for
enrichment of each cluster separately.

In [ ]:

source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)
library(OmnipathR)
library(dplyr)

## 1. Load the DMA results
For the same data for which we have prepared the metabolite ID feature space,
we will load now the differential metabolite results.
If you need to create differential results first you can use MetaProViz functionality:
https://saezlab.github.io/MetaProViz/reference/dma.html

In [ ]:

# Load data
data(tissue_dma) # compares tumour versus normal

## 2. Load the feature metadata
Prepared in 02_id_workflow and 03_id_to_pk

In [ ]:

# Load feature table
Tissue_MetaData_Extended_cleaned <- readRDS(file.path(mp_results_dir(), "MetaboliteIDs_expanded_cleanedKEGG.rds"))

# Combine with data
dma_res <- merge( # kegg.y is from old DMA
    x = Tissue_MetaData_Extended_cleaned,
    y = tissue_dma[,1:7],
    by = "Metabolite",
    all = TRUE
)

# Ensure unique IDs and full background --> we include measured features that
# do not have a KEGG ID.
dma_res_clean <-
    dma_res %>%
    select(
        Metabolite,
        KEGG,
        InputID_select,
        Log2FC,
        t.val,
        p.val,
        p.adj
    ) %>%
    mutate(  # make one KEGG column from "KEGG" and "InputID_select"
        KEGG = case_when(
            InputID_select == "KEGG" ~ KEGG,
            .default = InputID_select
        )
    ) %>%
    select(
        - InputID_select
    ) %>%
    mutate(  # mark original rows
        .row_id = row_number()
    ) %>%
    separate_rows(  # make long-format for cases with two KEGG IDs
        KEGG,
        sep = ",\\s*"
    ) %>%
    group_by( # create index per original row
        .row_id
    ) %>%
    mutate(
        .dup_index = row_number(),
        .n = n()
    ) %>%
    ungroup() %>%
    mutate(   #  only add suffix if there was splitting (n > 1)
        Metabolite = if_else(
            .n > 1,
            paste0(Metabolite, "_", .dup_index),
            Metabolite
        )
    ) %>%
    mutate(
        KEGG = if_else(
            is.na(KEGG),
            paste0("NA_", cumsum(is.na(KEGG))),
            KEGG
        )
    ) %>%
    group_by(
        KEGG
    ) %>%
    slice_max(    # For duplicated KEGG IDs --> keep higher absolute Log2FC
        order_by = Log2FC, n = 1, with_ties = FALSE
    ) %>%
    ungroup() %>%
    column_to_rownames(
        "KEGG"
    ) %>%
    rename(
        Metabolite_Name = Metabolite
    ) %>%
    select(
        - ".row_id",
        - ".dup_index",
        - ".n"
    )

# Inspect the dma_res_clean table

## 3. Load KEGG pathway-metabolite sets
We load the cleaned version via MetaProViz (part of MetSigDB)

In [ ]:
kegg_pw <- metsigdb_kegg()
head(kegg_pw)

# As we found some problematic cases we remove those from the PK
KEGG_Pathways_clean <-
    kegg_pw %>%
    filter(
        !(MetaboliteID == "C01087" & term == "Metabolic pathways"),  # Remove (R)-2-Hydroxyglutarate (C01087) from KEGG pathway "Metabolic Pathways"
        !(MetaboliteID == "C00668" & term %in% c(    # C00092, C00668: remove C00668 from Pathways which contain both of them
            "Metabolic pathways",
            "Biosynthesis of secondary metabolites"
        ))
    )

## Standard ORA per contrast

`standard_ora()` expects a table with `Metabolite` row names and
`t.val` / `p.adj` columns — exactly what `dma()` produces.

Note on ID matching: the toy dataset's `Metabolite` column uses
trivial names (e.g. `lactate`, `leucine`). Standard pathway resources
use HMDB or KEGG IDs. In a real workflow you'd translate first via
`OmnipathR::translate_ids()` or use a dataset-specific mapping. For
this demo we wrap the call in `tryCatch()` so an empty-overlap result
does not stop the script.

In [ ]:

set.seed(100)

Res <- standard_ora(
    data = dma_res_clean, # Input data requirements: column `t.val` and column `Metabolite`
    metadata_info = c(
        pvalColumn = "p.adj",
        percentageColumn = "t.val",
        PathwayTerm = "term",
        PathwayFeature = "MetaboliteID"
    ),
    input_pathway = KEGG_Pathways_clean,  # Pathway file requirements: column `term`, `Metabolite` and `Description`. Above we loaded the Kegg_Pathways using MetaProViz::Load_KEGG()
    pathway_name = "KEGG_TvN",
    min_gssize = 3,
    max_gssize = 1000,
    cutoff_stat = 0.01,
    cutoff_percentage = 10
)


# Select to plot:
Res_Select <- Res[["ClusterGosummary"]] %>%
    filter(p.adjust < 0.1)

# Plot Volcano of top altered pathways:
viz_volcano(
    plot_types = "PEA",
    data = dma_res_clean, # Must be the data you have used as an input for the pathway analysis
    data2 = as.data.frame(Res_Select) %>% dplyr::rename("term"="ID"),
    metadata_info = c(
        PEA_Pathway = "term",# Needs to be the same in both, metadata_feature and data2.
        PEA_stat = "p.adjust",#Column data2
        PEA_score = "GeneRatio",#Column data2
        PEA_Feature = "MetaboliteID"
    ), # Column metadata_feature (needs to be the same as row names in data)
    metadata_feature = KEGG_Pathways_clean, #Must be the pathways used for pathway analysis
    plot_name = "KEGG_TvN",
    select_label = NULL,
    subtitle = "PEA"
)

## Understand redundancy with `cluster_pk()`

Many pathway resources contain near-duplicate terms. `cluster_pk()`
clusters terms by their metabolite-set similarity (Jaccard,
overlap-coefficient, or correlation), then picks one representative
per cluster.

In [ ]:

cluster_pk_input <- Res[["ClusterGosummary"]] %>%
    filter(pvalue < 0.05)

cluster_pk_res <-
    cluster_pk(
        cluster_pk_input,
        metadata_info = c(
            metabolite_column = "Metabolites_in_pathway",
            pathway_column = "ID"
        ),
        input_format = "enrichment",
        similarity = "jaccard",
        threshold = 0.6,
        plot_threshold = 0.4,
        clust = "community",
        min = 1,
        node_size_column = "percentage_of_Pathway_detected",
        save_plot = NULL,
        plot_name = "Enrichment_example",
        print_plot = FALSE,
        min_degree = 0,
        show_density = TRUE,
        max_nodes = 1000
    )

head(cluster_pk_res$cluster_summary)
cluster_pk_res$graph_plot

set.seed(123)
viz_graph(
    similarity_matrix = cluster_pk_res$similarity_matrix,
    clusters = cluster_pk_res$clusters,
    plot_threshold = 0,
    plot_name = "Enrichment_example_plot_threshold_0",
    max_nodes = 1000,
    min_degree = 0,
    node_sizes = cluster_pk_res$node_sizes,
    show_density = TRUE,
    save_plot = "svg",
    print_plot = FALSE
)


# We could also do this using KEGG before we run enrichment analysis:
clustered_kegg <- cluster_pk(
    data = kegg_pw,
    metadata_info = c(metabolite_column = "MetaboliteID", pathway_column = "term"),
    similarity = "jaccard",
    threshold = 0.5,
    save_plot = NULL,
    print_plot = FALSE,
    path = mp_results_dir("05_enrichment")
)

names(clustered_kegg)
cluster_pk_input <- Res_Select



**Recap.** Three enrichment angles on the same dma table: raw KEGG ORA,
de-duplicated KEGG ORA via `cluster_pk()`. Then we
visualised one of these as a small network.

# Optional supplementary sections

# 6. Load the data and explore the SummarizedExperiment

We work with `intracell_raw_se`, a `SummarizedExperiment` shipped with
MetaProViz. It contains intracellular metabolomics from a kidney-cancer
study (Metabolomics Workbench `ST002224`): three ccRCC cell lines (786-O,
786-M1A, 786-M2A) and a non-cancerous proximal-tubule control (HK2), with
pool samples used for QC.

Since v3.99.10 (Oct 2025) MetaProViz uses `SummarizedExperiment` (SE) as
its canonical container — the same object Bioconductor's omics packages
expect. Each row is a metabolite; each column is a sample. Sample
annotations live in `colData(se)`, feature annotations in `rowData(se)`,
the actual measurements in `assays(se)`.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)
library(OmnipathR)

In [ ]:
data(intracell_raw_se)
intracell_raw_se

## What's in the SE?

In [ ]:
SummarizedExperiment::assayNames(intracell_raw_se)
dim(intracell_raw_se)

In [ ]:
# Sample metadata: condition, biological replicate, analytical replicate.
colData(intracell_raw_se) |> as.data.frame() |> head()

In [ ]:
# Feature metadata: only metabolite names at this stage.
rowData(intracell_raw_se) |> as.data.frame() |> head()

In [ ]:
# Raw peak values (rows = metabolites, columns = samples).
assay(intracell_raw_se)[1:6, 1:6]

## Pre-processing

`processing()` (was `PreProcessing()` before v3.0.1) takes one SE in,
returns one SE out. By default it does:

- **feature filtering** — drops metabolites detected in <80% of samples
  per condition (`featurefilt = "Modified"`)
- **TIC normalisation** — scales each sample by its total ion count
- **half-minimum imputation** of missing values
- **outlier detection** via Hotelling's T² (records outliers in
  `colData()`)

All choices are echoed to the log; QC plots are returned in the result
list and also saved under `MetaProViz_Results/` inside the working dir.

In [ ]:
processed <- processing(
    data = intracell_raw_se,
    metadata_info = c(
        Conditions = "Conditions",
        Biological_Replicates = "Biological_Replicates"
    ),
    featurefilt = "Modified",
    cutoff_featurefilt = 0.8,
    tic = TRUE,
    mvi = TRUE,
    save_plot = NULL,
    save_table = NULL,
    print_plot = FALSE,
    path = mp_results_dir("01_processing")
)

In [ ]:
# `processing()` returns a named list with $SE / $DF / $Plot, each itself
# a list keyed by stage (`data_Rawdata`, `Preprocessing_output`, …).
# We want the final stage.
processed_se <- processed$SE$Preprocessing_output
processed_se

In [ ]:
SummarizedExperiment::assayNames(processed_se)
table(processed_se$Outliers)

We drop pool samples (used for QC) and any flagged outliers before the
differential analysis in script 02.

In [ ]:
keep <- processed_se$Conditions != "Pool" & processed_se$Outliers == "no"
clean_se <- processed_se[, keep]
clean_se

In [ ]:
saveRDS(clean_se, file.path(mp_results_dir(), "intracell_clean_se.rds"))

**Recap.** We loaded a SE, ran one `processing()` call, dropped pools and
outliers, and saved the cleaned object. In script 02 we'll compute
differential abundance with `dma()`.

# 7. Differential metabolite analysis

We compare each ccRCC cell line (786-O, 786-M1A, 786-M2A) to HK2 with
`dma()` — MetaProViz's `D`ifferential `M`etabolite `A`nalysis routine.
`dma()` (formerly `DMA()`) consumes a `SummarizedExperiment` and returns
one SE per pairwise contrast plus diagnostic plots.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)

clean_se <- readRDS(file.path(mp_results_dir(), "intracell_clean_se.rds"))
clean_se

## Run `dma()`

`metadata_info` is the new snake_case parameter (was `MetadataInfo`). To
do all-vs-HK2 in one call, we set `Numerator = NULL` and
`Denominator = "HK2"` — every other condition becomes a contrast against
HK2. Stats engine is limma (`pval = "lmFit"`); FDR adjustment is BH.

In [ ]:
dma_results <- dma(
    data = clean_se,
    metadata_info = c(
        Conditions = "Conditions",
        Numerator = NULL,
        Denominator = "HK2"
    ),
    pval = "lmFit",
    padj = "fdr",
    save_plot = NULL,
    save_table = NULL,
    print_plot = FALSE,
    path = mp_results_dir("02_dma")
)

names(dma_results)

`dma_results` is a list with `$ShapiroTest`, `$BartlettTest`, `$dma`,
and `$VolcanoPlot`. The actual per-contrast rectangular results live in
`$dma`, keyed by contrast name.

In [ ]:
names(dma_results$dma)

In [ ]:
contrast <- "786-M1A_vs_HK2"
dma_table <- dma_results$dma[[contrast]]
head(dma_table)

## Volcano plot

`viz_volcano()` (was `VizVolcano()`) accepts the rectangular dma table
and uses sensible defaults; we only label a handful of metabolites.

In [ ]:
viz_volcano(
    plot_types = "Standard",
    data = dma_table |> tibble::column_to_rownames("Metabolite"),
    cutoff_x = 0.5,
    cutoff_y = 0.05,
    select_label = c("citrate", "succinate", "lactate", "alpha-ketoglutarate"),
    plot_name = contrast,
    subtitle = "Differential metabolites: 786-M1A vs HK2",
    save_plot = NULL,
    print_plot = FALSE,
    path = mp_results_dir("02_dma")
)

## Heatmap of the top differential metabolites

`viz_heatmap()` is the function where the X11 font error used to bite on
Ubuntu LTS. The repo's `.Rprofile` forces the cairo device, so this just
works.

In [ ]:
top_metabolites <- dma_table |>
    dplyr::arrange(p.adj) |>
    dplyr::slice_head(n = 30) |>
    dplyr::pull(Metabolite)

viz_heatmap(
    data = clean_se[top_metabolites, ],
    metadata_info = c(color = "Conditions"),
    plot_name = "Top 30 differential metabolites",
    save_plot = NULL,
    print_plot = FALSE,
    path = mp_results_dir("02_dma")
)

## Sample structure: PCA

A quick PCA confirms cell-line separation along PC1.

In [ ]:
viz_pca(
    data = clean_se,
    metadata_info = c(color = "Conditions"),
    plot_name = "PCA of cleaned intracellular metabolomics",
    save_plot = NULL,
    print_plot = FALSE,
    path = mp_results_dir("02_dma")
)

In [ ]:
saveRDS(dma_results, file.path(mp_results_dir(), "dma_results.rds"))

**Recap.** `dma()` produced three contrasts (`*_vs_HK2`). Volcano +
heatmap + PCA give us a quick read on biology before we move to
pathway-level interpretation in scripts 03-05.